# Battery Cell Manufacturing — Anomaly Detection
### Isolation Forest vs LSTM Autoencoder on Simulated 4680 Cell Sensor Data
**Harshita Mav · Syracuse University MS CS · Tesla Cell Manufacturing Interview**

---

## Problem Statement

Battery cell manufacturing (especially the 4680 form factor) involves highly precise, continuous processes:
- **Electrode coating** and **electrolyte filling** at high throughput
- **Formation cycling** to initialize cell chemistry
- **End-of-line testing** measuring voltage, resistance, capacity

Anomalies in sensor readings can indicate:
- Equipment drift or failure
- Out-of-spec cells that will fail in the field
- Process instabilities (temperature spikes, fill level deviations)

**Goal:** Compare two unsupervised anomaly detection approaches on time-series sensor data and identify which performs better for this domain.


## 1. Data Simulation

We simulate 4 key sensors from a 4680 cell production line at 10-second intervals over ~14 hours (5,000 samples).

| Sensor | Description | Normal Range |
|--------|-------------|-------------|
| `voltage_v` | Cell voltage during formation | ~3.65 V |
| `temperature_c` | Electrolyte fill temperature | ~25 °C |
| `internal_res_mohm` | Internal resistance post-formation | ~2.5 mΩ |
| `fill_level_ml` | Electrolyte fill volume | ~5.8 mL |

### Anomaly Types Injected
- **Point anomalies**: Instantaneous spikes (equipment glitch, sensor error)
- **Contextual anomalies**: Short windows of drift (process instability)
- **Collective anomalies**: Correlated multi-sensor deviations (systematic failure)

In [ ]:
import sys, os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

plt.style.use('dark_background')
COLORS = {'voltage': '#58a6ff', 'temperature': '#f78166', 'resistance': '#e3b341', 'fill': '#3fb950'}

from data.simulate import generate_sensor_data

df = generate_sensor_data(n_samples=5000)
print(f"Dataset shape: {df.shape}")
print(f"Anomaly rate: {df['label'].mean()*100:.1f}%")
print(f"\nAnomaly breakdown:")
print(df[df['label']==1]['anomaly_type'].value_counts())
df.describe()

In [ ]:
# Visualize raw sensor data with anomaly highlights
WINDOW = slice(0, 800)
dw = df.iloc[WINDOW]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.patch.set_facecolor('#0d1117')

sensors = [
    ('voltage_v',         'Cell Voltage (V)',         COLORS['voltage']),
    ('temperature_c',     'Temperature (°C)',         COLORS['temperature']),
    ('internal_res_mohm', 'Internal Resistance (mΩ)', COLORS['resistance']),
    ('fill_level_ml',     'Fill Level (mL)',          COLORS['fill']),
]

for ax, (col, ylabel, color) in zip(axes, sensors):
    ax.set_facecolor('#0d1117')
    ax.plot(dw['timestamp'], dw[col], color=color, lw=1, alpha=0.9, label='Normal')
    anom = dw[dw['label']==1]
    ax.scatter(anom['timestamp'], anom[col], color='#f78166', s=20, zorder=5, label='Anomaly', marker='x')
    ax.set_ylabel(ylabel, fontsize=9, color='#8b949e')
    ax.tick_params(colors='#8b949e', labelsize=8)
    for spine in ax.spines.values(): spine.set_color('#21262d')
    ax.grid(True, color='#21262d', lw=0.5, alpha=0.7)

axes[0].legend(loc='upper right', fontsize=9)
axes[-1].tick_params(axis='x', rotation=20)
fig.suptitle('4680 Cell Manufacturing — Sensor Data (first 800 samples)', 
             color='#f0f6fc', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sensor_overview.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 2. Model 1 — Isolation Forest

### Why Isolation Forest?
- **No labels required** — fully unsupervised, ideal for manufacturing where labeled anomaly data is scarce
- **Computationally efficient** — O(n log n), deployable in near-real-time
- **Explainable** — anomaly score is based on path length in random trees
- **Industry standard** for tabular sensor data anomaly detection

### How It Works
Isolation Forest isolates observations by randomly selecting a feature and a split value. Anomalies, being rare and different, require fewer splits to isolate — giving them shorter average path lengths and higher anomaly scores.

In [ ]:
from models.isolation_forest import train, predict, evaluate
from sklearn.metrics import classification_report, roc_auc_score

# Train
if_model, if_scaler = train(df, contamination=0.04)
df_if = predict(df, if_model, if_scaler)
if_metrics = evaluate(df_if)

print("Isolation Forest Results")
print("=" * 40)
for k, v in if_metrics.items():
    if k != 'report':
        print(f"  {k:15s}: {v}")

In [ ]:
# Plot IF anomaly scores
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.patch.set_facecolor('#0d1117')

for ax in [ax1, ax2]:
    ax.set_facecolor('#0d1117')
    ax.grid(True, color='#21262d', lw=0.5)
    for spine in ax.spines.values(): spine.set_color('#21262d')
    ax.tick_params(colors='#8b949e', labelsize=8)

dw_if = df_if.iloc[WINDOW]
ax1.plot(dw_if['timestamp'], dw_if['voltage_v'], color='#58a6ff', lw=1, alpha=0.8)
ax1.set_ylabel('Cell Voltage (V)', color='#8b949e', fontsize=9)
ax1.set_title('Isolation Forest — Anomaly Score vs Ground Truth', color='#f0f6fc', fontsize=11)

ax2.fill_between(dw_if['timestamp'], dw_if['if_anomaly_score'], alpha=0.5, color='#58a6ff')
ax2.plot(dw_if['timestamp'], dw_if['if_anomaly_score'], color='#58a6ff', lw=1)
ax2.axhline(0.5, color='#e3b341', lw=1, ls='--', label='Threshold=0.5')
anom_if = dw_if[dw_if['label']==1]
ax2.scatter(anom_if['timestamp'], anom_if['if_anomaly_score'], 
            color='#f78166', s=25, zorder=5, label='True Anomaly', marker='o')
ax2.set_ylabel('IF Anomaly Score', color='#8b949e', fontsize=9)
ax2.set_ylim(0, 1)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('if_scores.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 3. Model 2 — LSTM Autoencoder

### Why LSTM Autoencoder?
- **Temporal context** — LSTMs capture sequential dependencies across the 20-sample lookback window, making them superior for detecting **gradual drifts** and **contextual anomalies**
- **Reconstruction-based detection** — trained to reconstruct normal patterns; high reconstruction error = anomaly
- **No explicit anomaly labels needed** — train only on (mostly) normal data
- **Better for collective/contextual anomalies** than point-based models

### Architecture
```
Input (20 timesteps × 4 features)
    → LSTM(64) [Encoder]
    → RepeatVector(20) [Bottleneck]
    → LSTM(64, return_seq=True) [Decoder]
    → TimeDistributed(Dense(4)) [Reconstruction]
    → MSE Reconstruction Error → Anomaly Score
```

In [ ]:
# Note: TensorFlow is required for this cell.
# The Streamlit app includes a graceful fallback if TF is not installed.
try:
    from models.lstm_autoencoder import train as lstm_train, predict as lstm_predict, evaluate as lstm_evaluate
    lstm_model, lstm_scaler = lstm_train(df, epochs=25)
    df_lstm = lstm_predict(df, lstm_model, lstm_scaler)
    lstm_metrics = lstm_evaluate(df_lstm)
    print("LSTM Autoencoder Results")
    print("=" * 40)
    for k, v in lstm_metrics.items():
        if k != 'report':
            print(f"  {k:15s}: {v}")
except ImportError:
    print("TensorFlow not installed. Run: pip install tensorflow")
    print("The Streamlit dashboard includes a demo fallback mode.")

## 4. Model Comparison

| Dimension | Isolation Forest | LSTM Autoencoder |
|-----------|-----------------|------------------|
| **Anomaly type strength** | Point anomalies | Contextual & collective |
| **Training speed** | Fast (~seconds) | Slow (~minutes, GPU helps) |
| **Inference latency** | Very low | Low-medium |
| **Temporal awareness** | None | Yes (20-step window) |
| **Explainability** | Feature importance via path length | Reconstruction error per feature |
| **Production suitability** | Excellent baseline | Better for complex temporal patterns |
| **Data requirements** | Low (n~1000+) | Medium (n~5000+, mostly normal) |

### Recommendation for Tesla Cell Manufacturing:
- **Use Isolation Forest** as the always-on baseline for real-time point anomaly detection (low latency, interpretable)
- **Deploy LSTM Autoencoder** for batch re-analysis and catching subtle process drift that Isolation Forest misses
- **Ensemble both** with a voting threshold for critical quality gates (e.g., end-of-line testing)

In [ ]:
# ROC curves comparison
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.grid(True, color='#21262d', lw=0.5)
for spine in ax.spines.values(): spine.set_color('#21262d')
ax.tick_params(colors='#8b949e')

fpr, tpr, _ = roc_curve(df_if['label'], df_if['if_anomaly_score'])
ax.plot(fpr, tpr, color='#58a6ff', lw=2, label=f"Isolation Forest (AUC = {if_metrics['auc_roc']:.3f})")

# If LSTM results available
try:
    fpr2, tpr2, _ = roc_curve(df_lstm['label'], df_lstm['lstm_anomaly_score'])
    ax.plot(fpr2, tpr2, color='#3fb950', lw=2, label=f"LSTM Autoencoder (AUC = {lstm_metrics['auc_roc']:.3f})")
except: pass

ax.plot([0,1], [0,1], 'k--', color='#8b949e', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', color='#8b949e')
ax.set_ylabel('True Positive Rate', color='#8b949e')
ax.set_title('ROC Curve Comparison', color='#f0f6fc', fontsize=12)
ax.legend(fontsize=10, facecolor='#161b22', edgecolor='#21262d', labelcolor='#f0f6fc')
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Next Steps / Production Considerations

1. **Real Tesla data integration** — Replace simulation with actual formation cycling logs or end-of-line test data via data pipeline (Kafka → feature store → model)
2. **Online learning** — Retrain Isolation Forest periodically as process baseline shifts (equipment wear, seasonal temperature changes)
3. **Alerting integration** — Route high-confidence anomaly flags to MES (Manufacturing Execution System) or operator dashboard with cell ID, timestamp, affected sensors
4. **Root cause analysis** — Use SHAP values on Isolation Forest features to explain *which* sensor drove each anomaly flag
5. **Ensemble threshold tuning** — Optimize precision/recall tradeoff based on cost of false positives (unnecessary holds) vs false negatives (defective cells shipped)

---
*Harshita Mav · MS Computer Science, Syracuse University · Built for Tesla Cell Manufacturing Interview*